# W-BOT Piper TTS Eğitimi v2 — Fine-Tune (dfki-medium checkpoint'ten, genişletilmiş korpus)

**Çalıştırmadan önce:** Runtime → Change runtime type → **T4 GPU** (veya A100 varsa onu) seç.

Bu notebook, v1'in (320 cümle/13 dk, overfit şüphesi doğurmuştu — bkz.
PROJE_DURUMU.md "TTS — Piper Garble Fix + Telaffuz Sözlüğü + Checkpoint
Overfit Testi") **devamı değil, sıfırdan ayrı bir eğitim**: yeni, çok daha
büyük ve duygu/prozodi çeşitliliği hedefli bir korpusla (970 cümle — 694 v2 +
276 ek, ~65-85 dk, 17 farklı ton kategorisi: sıcak karşılama, heyecanlı öneri,
net onay, empatik özür, soru tonlaması, veda, alerji/uyarı, fiyat/sayı, fonetik
denge, uzun/karmaşık cümleler, yabancı-yazımlı kelimeler, şaşkınlık, ısrarlı-
sabırlı tekrar, kısa yanıtlar, mahrem ton, hızlı diyalog, karışık-duygu uzun
cümle) `tr_TR-dfki-medium` tabanından fine-tune başlatır.

⚠️ **v1 ile KARIŞTIRILMASIN:** Drive'da ayrı bir klasör (`wbot-tts-v2`)
kullanılıyor — v1'in `wbot-tts/checkpoints/` klasörüne (epoch 6799'a kadar
giden) dokunulmaz, resume mantığı yanlışlıkla oraya bakmaz.

⚠️ **"Taban checkpoint indir" hücresi çalıştırılmadan eğitim başlamaz** —
kendi önceki checkpoint'iniz yoksa script hata verip duracak, sessizce
sıfırdan başlamayacak.

## Google Drive Klasör Yapısı
Aşağıdaki klasörü Drive'ında oluştur ve dosyaları yükle:
```
MyDrive/
  wbot-tts-v2/
    dataset/
      wavs/          ← corpus_0001.wav … corpus_0694.wav +
                        ek_corpus_0001.wav … ek_corpus_0276.wav (970 dosya)
      metadata.csv   ← LJSpeech formatı
```

**metadata.csv** formatı (Windows'taki `data/piper_dataset_v2/metadata.csv` dosyası):
```
corpus_0001|Merhaba, hoş geldiniz! Sizi burada görmek çok güzel.|Merhaba, hoş geldiniz! Sizi burada görmek çok güzel.
```

---
**Oturumlar arası resume:** Her yeni oturumda sadece 1., 4., 4.5. ve 5.
(Train) hücrelerini çalıştır. Checkpoint'ler v1'deki gibi **her 100 epoch'ta
otomatik Drive'a kaydedilir** (Hücre 5, `sync_checkpoints` arka plan
thread'i, 5 dakikada bir kontrol) — oturum kopsa/kapansa bile kaybolmaz.


In [ ]:
# ── HÜCRE 1: GPU + Python kontrol ──────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'BULUNAMADI — Runtime türünü T4 olarak değiştir!')
print('Python:', sys.version)

if sys.version_info >= (3, 11):
    print('\nℹ  Kernel Python\'u 3.11+ — bu SORUN DEĞİL: Hücre 2, piper-phonemize')
    print('   için izole bir Python 3.10 venv kurup piper-train\'i orada çalıştırır.')
    print('   (Not: "Factory reset runtime" Colab\'ın kernel Python sürümünü')
    print('   DEĞİŞTİRMEZ — bu yüzden ayrı venv yaklaşımı kullanılıyor.)')
else:
    print('\n✓ Kernel Python\'u zaten 3.10 — Hücre 2 venv kurmadan doğrudan kullanabilir.')

In [ ]:
# ── HÜCRE 2: Kurulum (ilk oturumda bir kez çalıştır) ──────────────────────
# ÖNEMLİ: piper-phonemize'ın Linux için Python 3.12 wheel'i YOK (PyPI'de yalnızca
# macOS cp312 var — cp310/cp311 manylinux wheel'leri mevcut). Colab'ın kernel
# Python'u sürümü ne olursa olsun (genelde 3.12+), piper-train + piper-phonemize'ı
# İZOLE bir Python 3.10 venv içinde kurup çalıştırıyoruz; kernel Python'una
# dokunmuyoruz. Bu venv her fark Colab oturumunda (yeni /content) yeniden kurulur.
import os
import subprocess
import sys

PIPER_SRC = '/content/piper-src'
VENV_DIR  = '/content/venv310'
VENV_PY   = f'{VENV_DIR}/bin/python'


def run(cmd, desc, check=True):
    """Komutu çalıştırır, sonucu gösterir; check=True ise başarısızlıkta durur.
    (Not: eski script os.system() kullanıyordu — dönüş kodu HİÇ kontrol
    edilmiyordu, bu yüzden piper-phonemize kurulumu sessizce başarısız olup
    preprocess aşamasına kadar fark edilmemişti. Artık her adımın çıkış kodu
    kontrol ediliyor.)"""
    print(f'▶ {desc}...')
    result = subprocess.run(cmd, shell=isinstance(cmd, str),
                             capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ✗ HATA (kod {result.returncode}):')
        print((result.stderr or result.stdout)[-2000:])
        if check:
            raise RuntimeError(f'"{desc}" adımı başarısız oldu.')
    else:
        print(f'  ✓ {desc} tamam')
    return result


if not os.path.exists(VENV_PY):
    # Python 3.10 kur — Colab'ın kendi Ubuntu deposunda artık yok, deadsnakes PPA gerekiyor
    run('apt-get install -y software-properties-common > /dev/null 2>&1',
        'software-properties-common kur')
    run('add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1',
        'deadsnakes PPA ekle')
    run('apt-get update -qq', 'apt-get update')
    run('apt-get install -y python3.10 python3.10-venv python3.10-dev > /dev/null 2>&1',
        'python3.10 kur')
    run('apt-get install -y espeak-ng > /dev/null 2>&1', 'espeak-ng kur')

    # İzole venv oluştur
    run(f'python3.10 -m venv {VENV_DIR}', 'Python 3.10 venv oluştur')

    # pip'i düşür — pytorch-lightning 1.7.7'nin geçersiz "torch>=1.9.*"
    # metadata'sını yeni pip (>=24.1) reddediyor, eski pip görmezden geliyor.
    run(f'{VENV_PY} -m pip install -q -U "pip<24.1"', "pip<24.1 kur (venv içine)")

    # CUDA'lı torch — venv izole olduğu için Colab'ın sistem torch'unu miras almaz,
    # PyPI'nin varsayılan Linux x86_64 wheel'i CUDA runtime'ı zaten içeriyor.
    run(f'{VENV_PY} -m pip install -q torch torchvision torchaudio',
        "torch (CUDA'lı) kur — birkaç dakika sürebilir")

    # Diğer paketler — torchmetrics==0.11.4 pini GitHub #707'deki
    # "_compare_version" ImportError'ını önlemek için gerekli.
    run(f'{VENV_PY} -m pip install -q setuptools cython "numpy<2" six torchmetrics==0.11.4 '
        f'"pytorch-lightning~=1.7.0" "piper-phonemize~=1.1.0" '
        f'"librosa>=0.9.2,<1" "onnxruntime>=1.11.0"',
        # setuptools: pytorch-lightning 1.7.7 hala pkg_resources kullaniyor,
        # yeni Python venv'ler bunu artik otomatik getirmiyor (ModuleNotFoundError:
        # No module named 'pkg_resources' -- ayri, bilinen bir hata sinifi).
        'setuptools/cython/numpy/torchmetrics/pytorch-lightning/piper-phonemize/librosa/onnxruntime kur')

    # piper-train kaynaktan kur — --no-deps: requirements.txt'teki
    # "torch<2" pinini görmezden gel, venv'e kurduğumuz torch 2.x'i KORU.
    run(f'git clone -q https://github.com/rhasspy/piper.git {PIPER_SRC}',
        'piper reposunu klonla')
    run(f'{VENV_PY} -m pip install -q -e {PIPER_SRC}/src/python/ --no-deps',
        'piper-train kur (--no-deps)')

    # monotonic_align Cython extension derle — VENV_PY ile (kernel'in 3.12 ABI'siyle
    # DEĞİL, aksi halde derlenen .so venv'in 3.10'unda import edilemez)
    run(f'cd {PIPER_SRC}/src/python && {VENV_PY} '
        f'piper_train/vits/monotonic_align/setup.py build_ext --inplace',
        'monotonic_align derle')

    # __init__.py düzelt (nested import hatası)
    init_path = f'{PIPER_SRC}/src/python/piper_train/vits/monotonic_align/__init__.py'
    with open(init_path) as f:
        content = f.read()
    content = content.replace(
        'from .monotonic_align.core import maximum_path_c',
        'from .core import maximum_path_c'
    )
    with open(init_path, 'w') as f:
        f.write(content)
    print('✓ monotonic_align düzeltildi')

    # Sağlık kontrolü — venv İÇİNDE çalıştırılır (kernel'in kendi 3.12'sinde DEĞİL),
    # çünkü kontrol etmek istediğimiz tam olarak venv'in paket durumu.
    health = run([VENV_PY, '-c',
        'import torch, pytorch_lightning as pl, piper_train, piper_phonemize; '
        'print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | '
        'pytorch-lightning {pl.__version__}")'],
        'Sağlık kontrolü (venv içinde import)')
    print('  ' + health.stdout.strip())
    print('\n✅ Kurulum tamamlandı! İzole venv: ' + VENV_DIR)
else:
    print(f'✓ venv zaten kurulu: {VENV_DIR}')
    run(f'{VENV_PY} -m pip install -q -e {PIPER_SRC}/src/python/ --no-deps',
        'piper-train güncelle', check=False)


In [ ]:
# ── HÜCRE 3: Google Drive bağla + dosyaları /content'e kopyala ────────────
import os, shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT  = '/content/drive/MyDrive/wbot-tts-v2'
DATASET_SRC = f'{DRIVE_ROOT}/dataset'          # Drive'daki WAV + metadata.csv
DATASET_DST = '/content/piper-dataset'         # Colab hızlı SSD
PREPROC_DIR = '/content/piper-preprocessed'    # preprocess çıktısı
CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints'      # checkpoint'ler Drive'da saklanır

os.makedirs(CKPT_DIR, exist_ok=True)

# Dataset kontrol
if not os.path.exists(DATASET_SRC):
    raise FileNotFoundError(
        f'Drive\'da {DATASET_SRC} bulunamadı.\n'
        'MyDrive/wbot-tts-v2/dataset/ klasörünü oluştur ve '
        'wavs/ + metadata.csv dosyalarını yükle.'
    )

wav_count = len(list(os.scandir(f'{DATASET_SRC}/wavs'))) if os.path.exists(f'{DATASET_SRC}/wavs') else 0
print(f'✓ Drive bağlandı | Dataset: {wav_count} WAV dosyası')

# /content'e kopyala (Drive I/O yavaş, SSD hızlı)
if not os.path.exists(DATASET_DST):
    print('Dataset kopyalanıyor... (~30 sn)')
    shutil.copytree(DATASET_SRC, DATASET_DST)
    print(f'✓ {wav_count} WAV → {DATASET_DST}')
else:
    print(f'✓ Dataset zaten kopyalı: {DATASET_DST}')

In [ ]:
# ── HÜCRE 4: Preprocess (ilk oturumda bir kez çalıştır) ───────────────────
import os, subprocess, sys

PREPROC_DIR = '/content/piper-preprocessed'
DATASET_DST = '/content/piper-dataset'
DRIVE_ROOT  = '/content/drive/MyDrive/wbot-tts-v2'
PREPROC_DRIVE = f'{DRIVE_ROOT}/preprocessed'
VENV_PY     = '/content/venv310/bin/python'  # Hücre 2'deki izole Python 3.10 venv

if os.path.exists(f'{PREPROC_DIR}/dataset.jsonl'):
    print('✓ Preprocess zaten yapılmış, atlıyorum.')
else:
    # Drive'da önce önbellek var mı?
    if os.path.exists(f'{PREPROC_DRIVE}/dataset.jsonl'):
        print('Drive\'dan önbelleği kopyalıyorum...')
        import shutil
        shutil.copytree(PREPROC_DRIVE, PREPROC_DIR)
        print('✓ Preprocessed data kopyalandı.')
    else:
        print('Preprocess başlıyor (970 cümle, ~5-8 dk)...')
        # VENV_PY: piper-phonemize kernel'in Python'unda (3.12+) DEĞİL, Hücre 2'nin
        # kurduğu izole Python 3.10 venv'inde kurulu — piper_train çağrıları bu
        # yüzden hep VENV_PY ile yapılmalı, sys.executable ile DEĞİL.
        result = subprocess.run([
            VENV_PY, '-m', 'piper_train.preprocess',
            '--language', 'tr',
            '--input-dir', DATASET_DST,
            '--output-dir', PREPROC_DIR,
            '--dataset-format', 'ljspeech',
            '--single-speaker',
            '--sample-rate', '22050',
        ], capture_output=True, text=True)
        print(result.stdout[-2000:] if result.stdout else '')
        if result.returncode != 0:
            print('HATA:', result.stderr[-1000:])
        else:
            # Bir sonraki oturum için Drive'a kaydet
            import shutil
            shutil.copytree(PREPROC_DIR, PREPROC_DRIVE)
            print(f'✓ Preprocessed data Drive\'a kaydedildi: {PREPROC_DRIVE}')

# Sağlık kontrolü
import json
with open(f'{PREPROC_DIR}/dataset.jsonl') as f:
    first = json.loads(f.readline())
print(f'\nİlk örnek norm_path: {first["audio_norm_path"]}')
print(f'İlk örnek spec_path: {first["audio_spec_path"]}')

# Path'lerin /content'e işaret ettiğini doğrula
if not first['audio_norm_path'].startswith('/content'):
    print('\n⚠  Path\'ler /content\'e işaret etmiyor — düzeltiliyor...')
    import re
    with open(f'{PREPROC_DIR}/dataset.jsonl') as f:
        lines = f.readlines()
    fixed = []
    for line in lines:
        d = json.loads(line)
        for key in ('audio_path', 'audio_norm_path', 'audio_spec_path'):
            if key in d and not d[key].startswith('/'):
                d[key] = f'{PREPROC_DIR}/{d[key]}' if 'piper' in d[key] else f'{DATASET_DST}/{d[key]}'
        fixed.append(json.dumps(d, ensure_ascii=False))
    with open(f'{PREPROC_DIR}/dataset.jsonl', 'w') as f:
        f.write('\n'.join(fixed) + '\n')
    print('✓ Path\'ler düzeltildi.')
else:
    print('✓ Path\'ler doğru.')

In [ ]:
# ── HÜCRE 4.5: Taban checkpoint indir (fine-tune için) ─────────────────────
# Not: fahrettin-medium'un eğitim checkpoint'i (.ckpt) HuggingFace'ten
# kaldırılmış (piper-checkpoints deposunda "a voice was removed at the
# request of a user" commit'i — tr/tr_TR altında artık yalnızca dfki var).
# Bunun yerine tr_TR-dfki-medium checkpoint'i kullanılıyor: aynı dil/kalite/
# örnekleme hızı (22050 Hz), .ckpt formatında mevcut. Ses tonu önemli değil —
# fine-tune sırasında sizin 13 dakikalık kaydınızla ezilecek.
import os

DRIVE_ROOT    = '/content/drive/MyDrive/wbot-tts-v2'
BASE_CKPT_DIR = f'{DRIVE_ROOT}/base_checkpoint'
BASE_CKPT_URL = ('https://huggingface.co/datasets/rhasspy/piper-checkpoints/'
                  'resolve/main/tr/tr_TR/dfki/medium/'
                  'epoch%3D5679-step%3D1489110.ckpt')
BASE_CKPT_PATH = f'{BASE_CKPT_DIR}/tr_TR-dfki-medium.ckpt'

os.makedirs(BASE_CKPT_DIR, exist_ok=True)
if os.path.exists(BASE_CKPT_PATH) and os.path.getsize(BASE_CKPT_PATH) > 800_000_000:
    print(f'✓ Taban checkpoint zaten indirilmiş: {BASE_CKPT_PATH}')
else:
    print('Taban checkpoint indiriliyor (846 MB, birkaç dakika sürebilir)...')
    os.system(f'wget -q "{BASE_CKPT_URL}" -O "{BASE_CKPT_PATH}"')
    if os.path.exists(BASE_CKPT_PATH) and os.path.getsize(BASE_CKPT_PATH) > 800_000_000:
        size_mb = os.path.getsize(BASE_CKPT_PATH) / 1024 / 1024
        print(f'✓ İndirildi: {BASE_CKPT_PATH} ({size_mb:.0f} MB)')
    else:
        raise RuntimeError(
            'Checkpoint indirilemedi veya boyutu şüpheli küçük. '
            'URL değişmiş olabilir — HuggingFace deposunu kontrol edin: '
            'https://huggingface.co/datasets/rhasspy/piper-checkpoints/tree/main/tr/tr_TR/dfki/medium'
        )

In [ ]:
# ── HÜCRE 5: EĞİTİM (fine-tune) ────────────────────────────────────────────
# Resume için: önceki oturumda kalan son checkpoint otomatik bulunur.
# Yoksa dfki-medium taban checkpoint'inden fine-tune başlar (Hücre 4.5).
import os, glob, sys

PREPROC_DIR = '/content/piper-preprocessed'
DRIVE_ROOT  = '/content/drive/MyDrive/wbot-tts-v2'
CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints'
LOG_DIR     = f'{PREPROC_DIR}/lightning_logs'
VENV_PY     = '/content/venv310/bin/python'  # Hücre 2'deki izole Python 3.10 venv

# Drive'daki son checkpoint'i /content'e kopyala (resume için)
drive_ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.ckpt'))
resume_arg  = []
if drive_ckpts:
    last_ckpt = drive_ckpts[-1]
    local_ckpt = f'/content/resume.ckpt'
    import shutil
    shutil.copy2(last_ckpt, local_ckpt)
    resume_arg = ['--resume_from_checkpoint', local_ckpt]
    print(f'▶ Kendi checkpoint\'inizden devam: {os.path.basename(last_ckpt)}')
elif os.path.exists(BASE_CKPT_PATH):
    resume_arg = ['--resume_from_checkpoint', BASE_CKPT_PATH]
    print(f'▶ dfki-medium checkpoint\'inden fine-tune başlıyor: {BASE_CKPT_PATH}')
else:
    raise FileNotFoundError(
        'Ne kendi checkpoint\'iniz ne de dfki taban checkpoint\'i bulunamadı. '
        'Önce Hücre 4.5\'i ("Taban checkpoint indir") çalıştırın.'
    )

# VENV_PY: piper-train (ve piper-phonemize) kernel'in Python'unda değil,
# Hücre 2'nin kurduğu izole Python 3.10 venv'inde kurulu.
#
# Checkpoint callback: her 100 epoch'ta Drive'a kaydet
# (piper_train --checkpoint-epochs ile kontrol edilir)
cmd = [
    VENV_PY, '-m', 'piper_train',
    '--dataset-dir',       PREPROC_DIR,
    '--accelerator',       'gpu',
    '--devices',           '1',
    '--batch-size',        '32',     # A100 40GB için (T4 16GB varsayılanı 32'ye çıkarıldı — VRAM'e göre 48'e kadar denenebilir)
    '--validation-split',  '0.05',
    '--num-test-examples', '5',
    # Fine-tune: resmi Piper rehberi scratch için 2000, fine-tune için +1000
    # epoch öneriyor. Learning rate CLI'den ayarlanamıyor (piper_train
    # kaynağında sabit 2e-4, argparse'da yok) — epoch sayısını düşürmek
    # tek ayarlanabilir fine-tune sinyali.
    '--max_epochs',        '1000',
    '--checkpoint-epochs', '100',
    '--precision',         '32',
] + resume_arg

print('Eğitim başlıyor... (T4\'de fine-tune scratch\'e göre daha kısa sürer, progress bar aşağıda görünür)')
print(' '.join(cmd[:6]) + ' ...')

import subprocess, threading, time

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)

# Checkpoint'leri Drive'a kopyalayan arka plan iş parçacığı
def sync_checkpoints():
    while proc.poll() is None:
        time.sleep(300)  # 5 dakikada bir kontrol
        ckpts = sorted(glob.glob(f'{LOG_DIR}/**/epoch=*.ckpt', recursive=True))
        for ckpt in ckpts:
            dst = f'{CKPT_DIR}/{os.path.basename(ckpt)}'
            if not os.path.exists(dst):
                import shutil
                shutil.copy2(ckpt, dst)
                print(f'\n💾 Checkpoint kaydedildi: {os.path.basename(ckpt)}')

t = threading.Thread(target=sync_checkpoints, daemon=True)
t.start()

# Çıktıyı ekrana yansıt
for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\n\nEğitim tamamlandı. Return code: {proc.returncode}')

# Son checkpoint'i Drive'a kaydet
all_ckpts = sorted(glob.glob(f'{LOG_DIR}/**/epoch=*.ckpt', recursive=True))
if all_ckpts:
    import shutil
    for ckpt in all_ckpts:
        dst = f'{CKPT_DIR}/{os.path.basename(ckpt)}'
        if not os.path.exists(dst):
            shutil.copy2(ckpt, dst)
    print(f'✓ {len(all_ckpts)} checkpoint Drive\'a kaydedildi: {CKPT_DIR}')
else:
    print('⚠  Checkpoint bulunamadı.')

In [ ]:
# ── HÜCRE 6: ONNX Export ──────────────────────────────────────────────────
import os, glob, shutil, sys

PREPROC_DIR  = '/content/piper-preprocessed'
DRIVE_ROOT   = '/content/drive/MyDrive/wbot-tts-v2'
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoints'
MODEL_DIR    = f'{DRIVE_ROOT}/model'
LOG_DIR      = f'{PREPROC_DIR}/lightning_logs'
VENV_PY      = '/content/venv310/bin/python'  # Hücre 2'deki izole Python 3.10 venv

os.makedirs(MODEL_DIR, exist_ok=True)

# En son checkpoint'i bul
local_ckpts = sorted(glob.glob(f'{LOG_DIR}/**/epoch=*.ckpt', recursive=True))
drive_ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.ckpt'))
all_ckpts   = local_ckpts + drive_ckpts

if not all_ckpts:
    raise FileNotFoundError('Checkpoint bulunamadı. Önce eğitimi tamamla.')

# En yüksek epoch numaralı checkpoint
def epoch_num(p):
    import re
    m = re.search(r'epoch=(\d+)', os.path.basename(p))
    return int(m.group(1)) if m else -1

best_ckpt = max(all_ckpts, key=epoch_num)
print(f'Export edilecek checkpoint: {os.path.basename(best_ckpt)} (epoch {epoch_num(best_ckpt)})')

# Drive'daki checkpoint'i /content'e kopyala
local_ckpt = '/content/best.ckpt'
if best_ckpt.startswith(DRIVE_ROOT):
    shutil.copy2(best_ckpt, local_ckpt)
    best_ckpt = local_ckpt

# ONNX export — VENV_PY: piper-train, Hücre 2'nin kurduğu izole Python 3.10
# venv'inde kurulu, kernel'in kendi Python'unda değil.
ONNX_OUT = '/content/wbot_tr_v2.onnx'
result = os.system(
    f'"{VENV_PY}" -m piper_train.export_onnx '
    f'{best_ckpt} '
    f'{ONNX_OUT}'
)

if os.path.exists(ONNX_OUT):
    size_mb = os.path.getsize(ONNX_OUT) / 1024 / 1024
    print(f'\n✓ ONNX model: {ONNX_OUT} ({size_mb:.1f} MB)')

    # config.json kopyala (model'ın yanında olması gerekir)
    cfg_src = f'{PREPROC_DIR}/config.json'
    cfg_dst = '/content/wbot_tr_v2.onnx.json'
    shutil.copy2(cfg_src, cfg_dst)

    # Drive'a kaydet
    shutil.copy2(ONNX_OUT, f'{MODEL_DIR}/wbot_tr_v2.onnx')
    shutil.copy2(cfg_dst, f'{MODEL_DIR}/wbot_tr_v2.onnx.json')
    print(f'✓ Model Drive\'a kaydedildi: {MODEL_DIR}/')
else:
    print('Export başarısız.')

In [ ]:
# ── HÜCRE 7: Test (opsiyonel) ─────────────────────────────────────────────
import subprocess, IPython.display as ipd

TEST_TEXT = 'Merhaba, hoş geldiniz. Siparişinizi alıyorum.'
ONNX_PATH = '/content/wbot_tr_v2.onnx'
OUT_WAV   = '/content/test_output.wav'

# piper binary'yi indir (inference için)
if not os.path.exists('/content/piper/piper'):
    os.system('wget -q https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_linux_x86_64.tar.gz -O /tmp/piper.tar.gz')
    os.system('mkdir -p /content/piper && tar -xzf /tmp/piper.tar.gz -C /content/piper --strip-components=1')

result = subprocess.run(
    ['/content/piper/piper',
     '--model', ONNX_PATH,
     '--output_file', OUT_WAV],
    input=TEST_TEXT, text=True, capture_output=True
)

if os.path.exists(OUT_WAV):
    print(f'Test cümlesi: "{TEST_TEXT}"')
    ipd.display(ipd.Audio(OUT_WAV))
else:
    print('Test başarısız:', result.stderr)

In [ ]:
# ── HÜCRE 8: Model indir (Colab → bilgisayar) ─────────────────────────────
from google.colab import files
import os

for f in ['/content/wbot_tr_v2.onnx', '/content/wbot_tr_v2.onnx.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'İndiriliyor: {f}')
    else:
        print(f'Bulunamadı: {f} — önce export hücresini çalıştır')